# Orbit Wars Agent


In [ ]:
import kaggle_environments
print("kaggle_environments version:", kaggle_environments.__version__)
print("Offline-safe mode: using runtime-provided packages only.")

In [ ]:
%%writefile submission.py
import math
from typing import Dict, List, Tuple
from collections import namedtuple

Planet = namedtuple("Planet", ["id", "owner", "x", "y", "radius", "ships", "production"])

SUN_X, SUN_Y = 50.0, 50.0
SUN_RADIUS = 10.0
MAX_SPEED = 6.0
MIN_OFFENSE_SCORE = 0.035
MIN_SEND_SHIPS = 6
# Conquering: spam enemy planets in chunks; neutrals always get one-shot capture waves.
CONQUERING_SPAM_PER_WAVE = 28
# Neutral captures: split across multiple owned planets (primary sends more, others less).
MULTI_BASE_SPLIT_FLOOR = 16
PRIMARY_NEUTRAL_SHARE = 0.58
SECONDARY_NEUTRAL_SHARE = 0.38
# Expansion neutrals: avoid endless 6-ship taps; chunk floors until the flip.
EXPANSION_NEUTRAL_MIN_CHUNK_PRIMARY = 14
EXPANSION_NEUTRAL_MIN_CHUNK_SECONDARY = 12
EXPANSION_MAX_DISTINCT_NEUTRALS = 6
# Allow simultaneous nearby neutral grabs while still finishing committed closest target first.
EXPANSION_PARALLEL_NEARBY_DISTANCE = 4.0
# Expansion realism: avoid long-ETA chip sends that rarely convert before enemies contest.
EXPANSION_NEUTRAL_DECISIVE_MAX_ETA = 11
# Enemy planets: commit enough total ships (>= cost * factor) to beat production growth en route.
TAKEN_PRESSURE_FACTOR = 1.30
# Multi-base splits toward enemy-held: lower floor than neutrals so swarms start sooner.
TAKEN_MULTIBASE_SPLIT_FLOOR = 11
MAX_OFFENSE_COMMIT_RATIO_EXPANSION = 0.61
MAX_OFFENSE_COMMIT_RATIO_CONQUERING = 0.47
# Early turns + catch-up: prioritize neutrals and cheap grabs.
EXPANSION_MAX_STEP = 320
# Stay in expansion while small or still behind on ships / economy.
EXPANSION_MAX_PLANETS = 6
EXPANSION_PRODUCTION_RATIO = 0.92  # my_prod < enemy_prod * this => stay expansion after max_step
# While we own <= this many worlds, prefer capturing STATIC neutrals first (foothold vs risky orbiters).
STATIC_PRIORITY_MAX_BASES = 4
# Hard requirement: keep expanding until we have this much total production.
FOOTHOLD_MIN_TOTAL_PRODUCTION = 22
FOOTHOLD_NEUTRAL_MIN_PRODUCTION = 3
EXPANSION_SHIP_RATIO = 0.88  # my_ships < enemy_total * this => still expanding
# Re-scan sources in waves so one turn spends offense budget across multiple launches.
OFFENSE_WAVES_PER_TURN = 16
# Never plan to idle more than (1 - this) of pooled attack-eligible ships in a single turn.
OFFENSE_POOL_COMMIT_FLOOR = 0.67
DOMINANT_MODE_SHIP_ADVANTAGE = 1.65
# Force immediate expansion in opening ticks (do not wait to perfectly fund captures).
EARLY_FORCE_EXPANSION_STEP = 90
EARLY_ORBIT_BONUS_STEPS = 120  # boost grabbing closest dynamic planets early


def _effective_min_offense_score(step_now: int, expansion: bool) -> float:
    """Score gate; keep low enough that ships keep moving instead of banking."""
    if expansion:
        return MIN_OFFENSE_SCORE * 0.88
    if step_now >= 420:
        return MIN_OFFENSE_SCORE * 0.24
    if step_now >= 360:
        return MIN_OFFENSE_SCORE * 0.30
    if step_now >= 300:
        return MIN_OFFENSE_SCORE * 0.38
    if step_now >= 240:
        return MIN_OFFENSE_SCORE * 0.48
    if step_now >= 160:
        return MIN_OFFENSE_SCORE * 0.58
    return MIN_OFFENSE_SCORE * 0.68


def _distance(ax: float, ay: float, bx: float, by: float) -> float:
    return math.hypot(ax - bx, ay - by)


def _nearest_owned_distance(my_planets: List[Planet], t: Planet) -> float:
    return min(_distance(p.x, p.y, t.x, t.y) for p in my_planets)


def _neutral_take_cost(my_planets: List[Planet], t: Planet, step_now: int, my_player: int) -> float:
    """Estimate lowest practical neutral take cost from current owned planets."""
    best = float("inf")
    for src in my_planets:
        eta = _eta_turns(src, t, MIN_SEND_SHIPS)
        defenders = _project_defenders(t, eta, my_player)
        margin = _capture_margin(step_now=step_now, eta=eta, target=t, my_player=my_player)
        # Cost prioritizes cheap flips, then slightly prefers shorter ETAs.
        c = float(defenders + margin) + 0.45 * float(eta)
        if c < best:
            best = c
    return best


def _fleet_speed(ships: int, max_speed: float = MAX_SPEED) -> float:
    ships = max(1, int(ships))
    if ships <= 1:
        return 1.0
    scale = (math.log(ships) / math.log(1000)) ** 1.5
    scale = max(0.0, min(1.0, scale))
    return 1.0 + (max_speed - 1.0) * scale


def _segment_to_point_distance(ax: float, ay: float, bx: float, by: float, px: float, py: float) -> float:
    abx, aby = bx - ax, by - ay
    apx, apy = px - ax, py - ay
    ab_len2 = abx * abx + aby * aby
    if ab_len2 == 0.0:
        return math.hypot(apx, apy)
    t = (apx * abx + apy * aby) / ab_len2
    t = max(0.0, min(1.0, t))
    cx, cy = ax + t * abx, ay + t * aby
    return math.hypot(px - cx, py - cy)


def _line_hits_sun(src: Planet, dst: Planet, safety_margin: float = 0.4) -> bool:
    d = _segment_to_point_distance(src.x, src.y, dst.x, dst.y, SUN_X, SUN_Y)
    return d <= (SUN_RADIUS + safety_margin)


def _trajectory_hits_sun(src: Planet, angle: float, ships: int, max_ticks: int = 90) -> bool:
    speed = _fleet_speed(ships)
    x, y = src.x, src.y
    for _ in range(max_ticks):
        nx = x + math.cos(angle) * speed
        ny = y + math.sin(angle) * speed
        if _segment_to_point_distance(x, y, nx, ny, SUN_X, SUN_Y) <= SUN_RADIUS:
            return True
        x, y = nx, ny
        if x < -2 or x > 102 or y < -2 or y > 102:
            break
    return False


def _launch_intercepts_target(
    src: Planet,
    dst: Planet,
    angle: float,
    ships: int,
    step_now: int,
    angular_velocity: float,
    initial_by_id: Dict[int, Planet],
    max_ticks: int = 120,
) -> bool:
    speed = _fleet_speed(ships)
    x, y = src.x, src.y
    for tick in range(1, max_ticks + 1):
        nx = x + math.cos(angle) * speed
        ny = y + math.sin(angle) * speed

        # Out-of-bounds flights are considered failed launches.
        if nx < 0.0 or nx > 100.0 or ny < 0.0 or ny > 100.0:
            return False

        px, py = _predict_orbit_position(
            dst=dst,
            initial_by_id=initial_by_id,
            step_now=step_now,
            eta=tick,
            angular_velocity=angular_velocity,
        )
        if _segment_to_point_distance(x, y, nx, ny, px, py) <= (dst.radius + 0.22):
            return True

        x, y = nx, ny
    return False


def _safe_launch_angle(
    src: Planet,
    dst: Planet,
    send: int,
    step_now: int,
    angular_velocity: float,
    initial_by_id: Dict[int, Planet],
) -> float | None:
    base_angle = _angle_to_predicted(
        src=src,
        dst=dst,
        send=send,
        step_now=step_now,
        angular_velocity=angular_velocity,
        initial_by_id=initial_by_id,
    )

    # Try base angle first, then small corrections to recover near-misses.
    angle_candidates = [base_angle]
    for delta in (0.01, -0.01, 0.018, -0.018, 0.03, -0.03, 0.045, -0.045, 0.06, -0.06):
        angle_candidates.append(base_angle + delta)

    for angle in angle_candidates:
        if _trajectory_hits_sun(src, angle, send):
            continue
        if _launch_intercepts_target(
            src=src,
            dst=dst,
            angle=angle,
            ships=send,
            step_now=step_now,
            angular_velocity=angular_velocity,
            initial_by_id=initial_by_id,
        ):
            return angle
    return None


def _eta_turns(src: Planet, dst: Planet, ships: int) -> int:
    travel_dist = max(0.0, _distance(src.x, src.y, dst.x, dst.y) - src.radius - dst.radius)
    speed = _fleet_speed(ships)
    return max(1, math.ceil(travel_dist / max(1e-6, speed)))


def _project_defenders(target: Planet, eta: int, my_player: int) -> int:
    if target.owner == -1:
        return int(target.ships)
    if target.owner == my_player:
        return int(target.ships)
    return int(target.ships + target.production * eta)


def _capture_margin(step_now: int, eta: int, target: Planet, my_player: int) -> int:
    # Early game: small margins to expand quickly.
    # Mid/late game: higher margins to reduce near-miss failures.
    if step_now < 120:
        phase_margin = 1
    elif step_now < 300:
        phase_margin = 2
    else:
        phase_margin = 3

    eta_margin = min(4, max(0, eta // 7))
    prod_margin = 1 if target.owner >= 0 and target.owner != my_player else 0
    high_prod_margin = 1 if target.production >= 4 else 0
    return phase_margin + eta_margin + prod_margin + high_prod_margin


def _local_enemy_exposure(target: Planet, enemy_planets: List[Planet]) -> float:
    exposure = 0.0
    for enemy in enemy_planets:
        d = _distance(target.x, target.y, enemy.x, enemy.y)
        if d < 45.0:
            weight = (45.0 - d) / 45.0
            exposure += (enemy.ships + 2 * enemy.production) * weight
    return exposure


def _is_expansion_phase(
    step_now: int,
    my_planets: List[Planet],
    player: int,
    planets: List[Planet],
) -> bool:
    if step_now < EXPANSION_MAX_STEP:
        return True
    my_ships = sum(int(p.ships) for p in my_planets)
    enemy_ships = sum(int(p.ships) for p in planets if p.owner >= 0 and p.owner != player)
    my_prod = sum(int(p.production) for p in my_planets)
    enemy_prod = sum(int(p.production) for p in planets if p.owner >= 0 and p.owner != player)
    if len(my_planets) <= EXPANSION_MAX_PLANETS:
        return True
    if enemy_ships > 0 and my_ships < enemy_ships * EXPANSION_SHIP_RATIO:
        return True
    if enemy_prod > 0 and my_prod < enemy_prod * EXPANSION_PRODUCTION_RATIO:
        return True
    return False


def _planet_is_orbiting(dst: Planet, initial_by_id: Dict[int, Planet]) -> bool:
    ip = initial_by_id.get(dst.id)
    return ip is not None and _is_orbiting(ip)


def _target_score_v2(
    src: Planet,
    dst: Planet,
    player: int,
    trial_send: int,
    eta: int,
    defenders: int,
    enemy_planets: List[Planet],
    expansion: bool,
    initial_by_id: Dict[int, Planet],
    step_now: int,
) -> float:
    d = _distance(src.x, src.y, dst.x, dst.y)
    if d <= 0.0:
        return -1e9

    orbit_bonus = 0.0
    value_gain = 8.0 * float(dst.production)
    owner_bonus = 6.0 if dst.owner >= 0 and dst.owner != player else 0.0
    neutral_bonus = 2.0 if dst.owner == -1 else 0.0
    if dst.owner == -1 and dst.production >= 4:
        neutral_bonus += 4.5

    late_pressure = 0.0
    if not expansion and dst.owner >= 0 and dst.owner != player:
        if step_now >= 340:
            late_pressure = 10.5 + 1.35 * float(dst.production)
        elif step_now >= 260:
            late_pressure = 7.0 + 1.1 * float(dst.production)
        elif step_now >= 200:
            late_pressure = 5.0 + 0.85 * float(dst.production)
        elif step_now >= 120:
            late_pressure = 2.6 + 0.58 * float(dst.production)

    capture_cost = 0.30 * defenders
    distance_cost = 0.11 * d
    eta_cost = 0.65 * eta
    exposure_cost = 0.06 * _local_enemy_exposure(dst, enemy_planets)

    # Conquering: favor reachable enemy worlds (closest + lower ETA) for sustained pressure.
    if not expansion and dst.owner >= 0 and dst.owner != player:
        distance_cost *= 0.82
        eta_cost *= 0.90
        # Threat weighting: remove productive nearby enemy worlds earlier.
        threat_weight = 1.0 + max(0.0, (34.0 - d) / 34.0)
        value_gain += 3.8 * float(dst.production) * threat_weight

    # Penalize huge over-send by this source.
    oversend_cost = 0.20 * max(0.0, float(trial_send - defenders))

    if expansion:
        # Expansion: neutrals and fast grabs; de-prioritize costly enemy strikes.
        if dst.owner == -1:
            value_gain *= 1.18
            neutral_bonus *= 1.35
            # Heavily proximity weighted: closest opens should win races consistently.
            distance_cost *= 0.32
            eta_cost *= 0.62
            value_gain += max(0.0, 14.0 - 0.55 * d)
        elif dst.owner >= 0 and dst.owner != player:
            owner_bonus *= 0.35
            capture_cost *= 1.25
            exposure_cost *= 1.35
            eta_cost *= 1.12

    # Orbiting neutrals: small early nudge; later keep distance-driven race to opens.
    if dst.owner == -1 and _planet_is_orbiting(dst, initial_by_id):
        orbit_bonus = 4.2 if step_now < EARLY_ORBIT_BONUS_STEPS else 0.9
        distance_cost *= 0.90

    return (
        value_gain
        + owner_bonus
        + neutral_bonus
        + orbit_bonus
        + late_pressure
        - capture_cost
        - distance_cost
        - eta_cost
        - exposure_cost
        - oversend_cost
    )


def _angle_to(src: Planet, dst: Planet) -> float:
    return math.atan2(dst.y - src.y, dst.x - src.x)


def _is_orbiting(initial_p: Planet) -> bool:
    orbital_radius = _distance(initial_p.x, initial_p.y, SUN_X, SUN_Y)
    return (orbital_radius + initial_p.radius) < 50.0


def _predict_orbit_position(
    dst: Planet,
    initial_by_id: Dict[int, Planet],
    step_now: int,
    eta: int,
    angular_velocity: float,
) -> Tuple[float, float]:
    initial_p = initial_by_id.get(dst.id)
    if initial_p is None or not _is_orbiting(initial_p):
        return dst.x, dst.y

    # Use initial position + global angular velocity to estimate future position.
    r = _distance(initial_p.x, initial_p.y, SUN_X, SUN_Y)
    start_angle = math.atan2(initial_p.y - SUN_Y, initial_p.x - SUN_X)
    future_angle = start_angle + angular_velocity * (step_now + eta)
    return SUN_X + r * math.cos(future_angle), SUN_Y + r * math.sin(future_angle)


def _angle_to_predicted(
    src: Planet,
    dst: Planet,
    send: int,
    step_now: int,
    angular_velocity: float,
    initial_by_id: Dict[int, Planet],
) -> float:
    # Two-pass refinement: estimate ETA, predict position, recompute ETA once.
    eta1 = _eta_turns(src, dst, send)
    px1, py1 = _predict_orbit_position(dst, initial_by_id, step_now, eta1, angular_velocity)
    pseudo1 = Planet(dst.id, dst.owner, px1, py1, dst.radius, dst.ships, dst.production)
    eta2 = _eta_turns(src, pseudo1, send)
    px2, py2 = _predict_orbit_position(dst, initial_by_id, step_now, eta2, angular_velocity)
    return math.atan2(py2 - src.y, px2 - src.x)


def _dynamic_reserve(
    src: Planet,
    my_planets: List[Planet],
    enemy_planets: List[Planet],
    step_now: int,
) -> int:
    # Base reserve scales with production so stronger planets keep more defenders.
    base_reserve = max(5, 3 + 2 * int(src.production))
    if not enemy_planets:
        out = base_reserve
    else:
        min_enemy_dist = float("inf")
        local_pressure = 0.0
        for enemy in enemy_planets:
            d = _distance(src.x, src.y, enemy.x, enemy.y)
            if d < min_enemy_dist:
                min_enemy_dist = d
            if d < 35.0:
                weight = (35.0 - d) / 35.0
                local_pressure += (enemy.ships + 2 * enemy.production) * weight

        proximity_bonus = max(0, int((22.0 - min_enemy_dist) / 3.0))
        pressure_bonus = int(local_pressure / 8.0)
        fragile_empire_bonus = 3 if len(my_planets) <= 2 else 0
        out = base_reserve + proximity_bonus + pressure_bonus + fragile_empire_bonus

    # Established empire late game: bank less per planet so offense can keep pressure.
    if step_now >= 320 and len(my_planets) >= 6:
        out = max(int(out * 0.91), base_reserve + 1)
    elif step_now >= 280 and len(my_planets) >= 5:
        out = max(int(out * 0.94), base_reserve + 2)
    if step_now >= 180:
        out = max(base_reserve + 1, int(out * 0.93))
    return out


def _defense_first_reinforcements(
    my_planets: List[Planet],
    available_by_source: Dict[int, int],
    reserve_by_source: Dict[int, int],
    step_now: int,
    angular_velocity: float,
    initial_by_id: Dict[int, Planet],
) -> List[List[float]]:
    moves: List[List[float]] = []

    threatened: List[Tuple[int, Planet]] = []
    for p in my_planets:
        deficit = max(0, reserve_by_source[p.id] - int(p.ships))
        if deficit > 0:
            threatened.append((deficit, p))
    threatened.sort(key=lambda x: x[0], reverse=True)

    for deficit, target in threatened:
        if deficit <= 0:
            continue

        donors = []
        for donor in my_planets:
            if donor.id == target.id:
                continue
            avail = available_by_source.get(donor.id, 0)
            if avail <= 0:
                continue
            d = _distance(donor.x, donor.y, target.x, target.y)
            if d > 42.0:
                continue
            donors.append((d, donor))
        donors.sort(key=lambda x: x[0])

        for _, donor in donors:
            avail = available_by_source.get(donor.id, 0)
            if avail <= 0 or deficit <= 0:
                continue
            send = min(avail, deficit)
            if send < 4:
                continue
            if _line_hits_sun(donor, target):
                continue

            angle = _safe_launch_angle(
                src=donor,
                dst=target,
                send=send,
                step_now=step_now,
                angular_velocity=angular_velocity,
                initial_by_id=initial_by_id,
            )
            if angle is None:
                continue
            moves.append([int(donor.id), float(angle), int(send)])
            available_by_source[donor.id] = avail - send
            deficit -= send

    return moves


def agent(obs):
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    raw_initial_planets = obs.get("initial_planets", []) if isinstance(obs, dict) else obs.initial_planets
    step_now = obs.get("step", 0) if isinstance(obs, dict) else obs.step
    angular_velocity = obs.get("angular_velocity", 0.0) if isinstance(obs, dict) else obs.angular_velocity
    planets = [Planet(*p) for p in raw_planets]
    initial_by_id: Dict[int, Planet] = {p[0]: Planet(*p) for p in raw_initial_planets}

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    enemy_planets = [p for p in planets if p.owner >= 0 and p.owner != player]
    if not my_planets or not targets:
        return []

    expansion = _is_expansion_phase(step_now, my_planets, player, planets)
    # Strict expansion while untaken neutrals exist; once none remain, commit fully to enemy offense.
    neutral_only = any(p.owner == -1 for p in planets)

    commit_ratio = MAX_OFFENSE_COMMIT_RATIO_EXPANSION if neutral_only else MAX_OFFENSE_COMMIT_RATIO_CONQUERING
    if not neutral_only:
        if step_now >= 320:
            commit_ratio = min(0.70, commit_ratio + 0.26)
        elif step_now >= 260:
            commit_ratio = min(0.64, commit_ratio + 0.22)
        elif step_now >= 220:
            commit_ratio = min(0.60, commit_ratio + 0.16)

    effective_min_score = _effective_min_offense_score(step_now, neutral_only)
    my_total_production = sum(int(p.production) for p in my_planets)
    my_ships_total = sum(int(p.ships) for p in my_planets)
    enemy_ships_total = sum(int(p.ships) for p in enemy_planets)
    ship_advantage = my_ships_total / max(1, enemy_ships_total)
    foothold_mode = neutral_only and my_total_production < FOOTHOLD_MIN_TOTAL_PRODUCTION
    # Ahead on fleet: soften score gates (not "risk model" — raw thresholds were starving attacks).
    score_relief = max(0.30, 1.0 - min(0.70, max(0.0, (ship_advantage - 1.06) * 0.52)))
    effective_min_score *= score_relief
    # Raise per-turn pressure quota on enemy worlds when we out-mass them so we don't stop after first quota fill.
    taken_pressure_scale = 1.0 + min(0.75, max(0.0, (ship_advantage - 1.04) * 0.55))
    dominant_mode = (not neutral_only) and ship_advantage >= DOMINANT_MODE_SHIP_ADVANTAGE
    early_force_expansion = neutral_only and step_now <= EARLY_FORCE_EXPANSION_STEP
    if early_force_expansion:
        commit_ratio = max(commit_ratio, 0.92)
        effective_min_score *= 0.72
    if dominant_mode:
        commit_ratio = max(commit_ratio, 0.82)
        effective_min_score *= 0.70
        taken_pressure_scale *= 1.15

    reserve_by_source: Dict[int, int] = {
        p.id: _dynamic_reserve(p, my_planets, enemy_planets, step_now) for p in my_planets
    }
    available_by_source: Dict[int, int] = {
        p.id: max(0, int(p.ships) - reserve_by_source[p.id]) for p in my_planets
    }

    target_by_id: Dict[int, Planet] = {t.id: t for t in targets}
    target_allocated: Dict[int, int] = {t.id: 0 for t in targets}
    moves: List[List[float]] = _defense_first_reinforcements(
        my_planets=my_planets,
        available_by_source=available_by_source,
        reserve_by_source=reserve_by_source,
        step_now=step_now,
        angular_velocity=angular_velocity,
        initial_by_id=initial_by_id,
    )

    # Directional offense: expansion spreads across many neutrals with deliberate send sizes; conquering stays chunky.
    min_packet = MIN_SEND_SHIPS
    offense_candidates = [
        p for p in my_planets if available_by_source.get(p.id, 0) >= min_packet
    ]
    pool_offense = sum(available_by_source.get(p.id, 0) for p in offense_candidates)
    total_offense_budget = int(pool_offense * commit_ratio)
    total_offense_budget = min(
        pool_offense,
        max(total_offense_budget, int(pool_offense * OFFENSE_POOL_COMMIT_FLOOR)),
    )
    offense_sent_so_far = 0
    attacked_target_ids = set()

    neutral_targets = [p for p in targets if p.owner == -1]
    offense_targets = neutral_targets if neutral_only and neutral_targets else targets

    early_static_priority = (
        neutral_only
        and offense_targets
        and len(my_planets) <= STATIC_PRIORITY_MAX_BASES
    )

    # Neutral expansion priority: lowest capture-cost untakens first, then nearest follow-ups.
    if neutral_only and offense_targets:
        targets_rr_base = sorted(
            offense_targets,
            key=lambda t: (
                _neutral_take_cost(my_planets, t, step_now, player),
                _nearest_owned_distance(my_planets, t),
                t.id,
            ),
        )
    else:
        # Neutrals gone or already taking enemies: nearest threats first, then production tie-break.
        targets_rr_base = sorted(
            offense_targets,
            key=lambda t: (
                _nearest_owned_distance(my_planets, t),
                -t.production,
                t.id,
            ),
        )

    # Strongest owned planet (ships + production): main base for split sends toward neutrals.
    primary_base_id = max(
        my_planets,
        key=lambda p: int(p.ships) + 4 * int(p.production),
    ).id

    for wave in range(OFFENSE_WAVES_PER_TURN):
        if offense_sent_so_far >= total_offense_budget:
            break

        # Prefer finishing committed neutrals first, but keep other nearby openings available
        # so we don't idle while waiting on unrelated in-flight captures.
        if neutral_only and offense_targets:
            if len(my_planets) >= 2:
                pool = offense_targets
                targets_rr = sorted(
                    pool,
                    key=lambda t: (
                        -(target_allocated[t.id] > 0),
                        _neutral_take_cost(my_planets, t, step_now, player),
                        _nearest_owned_distance(my_planets, t),
                        t.id,
                    ),
                )
            else:
                home = my_planets[0]
                targets_rr = sorted(
                    offense_targets,
                    key=lambda t: (
                        -(target_allocated[t.id] > 0),
                        _neutral_take_cost(my_planets, t, step_now, player),
                        _distance(home.x, home.y, t.x, t.y),
                        t.id,
                    ),
                )
        else:
            targets_rr = list(targets_rr_base)
        if targets_rr and not (neutral_only and len(my_planets) == 1):
            rot = (step_now + wave) % len(targets_rr)
            targets_rr = targets_rr[rot:] + targets_rr[:rot]

        offensive_sources = [
            p for p in my_planets if available_by_source.get(p.id, 0) >= min_packet
        ]
        if not offensive_sources:
            break
        if not neutral_only:
            # Orbiting bases first (often closest to contested arc), then largest available stacks.
            offensive_sources.sort(
                key=lambda p: (
                    int(not _planet_is_orbiting(p, initial_by_id)),
                    -available_by_source[p.id],
                )
            )
        else:
            offensive_sources.sort(key=lambda p: available_by_source[p.id], reverse=True)

        if neutral_only:
            if len(my_planets) == 1:
                max_distinct_targets = 1
            else:
                max_distinct_targets = min(
                    EXPANSION_MAX_DISTINCT_NEUTRALS,
                    max(3, min(6, len(offensive_sources) + 1)),
                )
        elif expansion:
            max_distinct_targets = max(7, min(11, len(offensive_sources) + 4))
        else:
            max_distinct_targets = max(6, min(14, len(offensive_sources) // 2 + 6))

        for src in offensive_sources:
            if offense_sent_so_far >= total_offense_budget:
                break
            available = available_by_source[src.id]
            if available < min_packet:
                continue

            best_choice = None
            best_score = -1.0
            passes = (0, 1, 2, 3) if ship_advantage >= 1.14 else (0, 1, 2)
            for pass_ix in passes:
                if pass_ix == 0:
                    thresh = effective_min_score
                elif pass_ix == 1:
                    thresh = max(0.005, effective_min_score * 0.38)
                elif pass_ix == 2:
                    thresh = max(0.002, effective_min_score * 0.10)
                else:
                    thresh = max(0.0, effective_min_score * 0.03)
                # Conquering pass 1: retry focusing neutrals with a softer gate.
                neutral_pass = not neutral_only and pass_ix == 1
                for dst in targets_rr:
                    if neutral_pass and dst.owner != -1:
                        continue
                    if neutral_only and dst.owner >= 0:
                        continue
                    # Closest-first expansion per SOURCE with controlled nearby parallelism:
                    # - always allow already-committed targets (finish what we started),
                    # - otherwise constrain each source to its own nearest neutral neighborhood.
                    if neutral_only and targets_rr:
                        dst_is_committed = target_allocated.get(dst.id, 0) > 0
                        if not dst_is_committed:
                            uncommitted = [x for x in targets_rr if target_allocated.get(x.id, 0) <= 0]
                            if uncommitted:
                                nearest_src_dist = min(_distance(src.x, src.y, x.x, x.y) for x in uncommitted)
                                dst_src_dist = _distance(src.x, src.y, dst.x, dst.y)
                                allow_nearby = dst_src_dist <= (nearest_src_dist + EXPANSION_PARALLEL_NEARBY_DISTANCE)
                                if not (allow_nearby or early_force_expansion):
                                    continue
                    if _line_hits_sun(src, dst):
                        continue

                    remaining_budget = max(0, total_offense_budget - offense_sent_so_far)
                    if remaining_budget < min_packet:
                        continue

                    untaken = dst.owner == -1
                    send_work = max(min_packet, min(available, remaining_budget))
                    chosen = 0

                    for _ in range(14):
                        eta = _eta_turns(src, dst, send_work)
                        defenders = _project_defenders(dst, eta, player)
                        margin = _capture_margin(step_now=step_now, eta=eta, target=dst, my_player=player)
                        need_total = defenders + margin
                        remaining_need = need_total - target_allocated[dst.id]
                        if remaining_need <= 0:
                            chosen = 0
                            break
                        need_cap = min(remaining_need, available, remaining_budget)
                        if need_cap < MIN_SEND_SHIPS:
                            chosen = 0
                            break

                        # In strict neutral expansion, skip long-ETA partial chips that don't secure a flip.
                        if (
                            neutral_only
                            and untaken
                            and eta > EXPANSION_NEUTRAL_DECISIVE_MAX_ETA
                            and need_cap < remaining_need
                        ):
                            chosen = 0
                            break

                        # Avoid "tiny chip" sends (often landing at the MIN_SEND_SHIPS floor)
                        # when we can't contribute a meaningful chunk toward a same-turn capture.
                        if neutral_only and untaken:
                            # Opening: allow early foothold launches even if one-turn full cover is not yet possible.
                            if (not early_force_expansion) and remaining_budget < remaining_need:
                                chosen = 0
                                break

                            min_chunk = (
                                EXPANSION_NEUTRAL_MIN_CHUNK_PRIMARY
                                if src.id == primary_base_id
                                else EXPANSION_NEUTRAL_MIN_CHUNK_SECONDARY
                            )
                            # Only enforce chunk floor outside opening force-expansion window.
                            if (not early_force_expansion) and remaining_need >= min_chunk and need_cap < min_chunk:
                                chosen = 0
                                break

                        if untaken:
                            # Deliberate neutral capture quantities: finish quickly, no micro-spam.
                            if neutral_only and len(my_planets) == 1:
                                target_send = min(need_cap, remaining_need)
                            elif (
                                neutral_only
                                and len(my_planets) >= 2
                                and remaining_need >= MULTI_BASE_SPLIT_FLOOR
                            ):
                                praw = int(remaining_need * PRIMARY_NEUTRAL_SHARE + 0.5)
                                sraw = int(remaining_need * SECONDARY_NEUTRAL_SHARE + 0.5)
                                if src.id == primary_base_id:
                                    chunk = max(
                                        EXPANSION_NEUTRAL_MIN_CHUNK_PRIMARY,
                                        praw,
                                        int(remaining_need * 0.42),
                                    )
                                else:
                                    chunk = max(
                                        EXPANSION_NEUTRAL_MIN_CHUNK_SECONDARY,
                                        sraw,
                                        int(remaining_need * 0.30),
                                    )
                                chunk = min(chunk, remaining_need)
                                target_send = min(need_cap, chunk, remaining_need)
                            else:
                                target_send = min(need_cap, remaining_need)
                                if neutral_only and remaining_need >= EXPANSION_NEUTRAL_MIN_CHUNK_SECONDARY:
                                    target_send = max(
                                        target_send,
                                        min(remaining_need, EXPANSION_NEUTRAL_MIN_CHUNK_SECONDARY),
                                    )
                                    target_send = min(target_send, need_cap, remaining_need)
                        else:
                            # Taken: total ships assigned this turn toward dst should reach >= factor * projected cost.
                            pressure_quota = int(
                                math.ceil(need_total * TAKEN_PRESSURE_FACTOR * taken_pressure_scale)
                            )
                            # Prevent low-value overkill while still allowing heavier finish when dominant.
                            if dominant_mode:
                                overkill_cap = need_total + max(12, int(need_total * 0.75))
                            else:
                                overkill_cap = need_total + max(8, int(need_total * 0.35))
                            pressure_quota = min(pressure_quota, overkill_cap)
                            still_pressure = max(0, pressure_quota - target_allocated[dst.id])
                            if still_pressure <= 0:
                                chosen = 0
                                break
                            need_p = min(still_pressure, available, remaining_budget)
                            if need_p < MIN_SEND_SHIPS:
                                chosen = 0
                                break
                            if len(my_planets) >= 2 and still_pressure >= TAKEN_MULTIBASE_SPLIT_FLOOR:
                                if src.id == primary_base_id:
                                    chunk = max(
                                        MIN_SEND_SHIPS,
                                        int(still_pressure * PRIMARY_NEUTRAL_SHARE + 0.5),
                                    )
                                else:
                                    chunk = max(
                                        MIN_SEND_SHIPS,
                                        int(still_pressure * SECONDARY_NEUTRAL_SHARE + 0.5),
                                    )
                                target_send = min(need_p, chunk, still_pressure)
                            else:
                                spam = max(
                                    MIN_SEND_SHIPS,
                                    min(still_pressure, CONQUERING_SPAM_PER_WAVE),
                                )
                                target_send = min(need_p, spam, still_pressure)

                        if send_work == target_send:
                            chosen = target_send
                            send_work = target_send
                            break
                        send_work = target_send

                    if chosen < MIN_SEND_SHIPS:
                        continue

                    eta = _eta_turns(src, dst, send_work)
                    defenders = _project_defenders(dst, eta, player)
                    margin = _capture_margin(step_now=step_now, eta=eta, target=dst, my_player=player)
                    need_total = defenders + margin
                    remaining_need = need_total - target_allocated[dst.id]
                    need_cap = min(remaining_need, available, remaining_budget)
                    if remaining_need <= 0 or need_cap <= 0:
                        continue
                    send = min(send_work, need_cap)
                    if send < MIN_SEND_SHIPS:
                        continue

                    # Prevent spraying across too many targets in one turn.
                    if dst.id not in attacked_target_ids and len(attacked_target_ids) >= max_distinct_targets:
                        continue

                    trial_send = send
                    base_score = _target_score_v2(
                        src=src,
                        dst=dst,
                        player=player,
                        trial_send=trial_send,
                        eta=eta,
                        defenders=defenders,
                        enemy_planets=enemy_planets,
                        expansion=neutral_only,
                        initial_by_id=initial_by_id,
                        step_now=step_now,
                    )
                    saturation_penalty = 1.0 + (target_allocated[dst.id] / max(1.0, need_total))
                    saturation_penalty = min(2.0, saturation_penalty)
                    directional_score = base_score / saturation_penalty
                    if directional_score < thresh:
                        continue

                    if directional_score > best_score:
                        best_score = directional_score
                        best_choice = (dst, send)

                if best_choice is not None:
                    break

            if best_choice is None:
                continue

            dst, send = best_choice
            angle = _safe_launch_angle(
                src=src,
                dst=dst,
                send=send,
                step_now=step_now,
                angular_velocity=angular_velocity,
                initial_by_id=initial_by_id,
            )
            if angle is None:
                continue
            moves.append([int(src.id), float(angle), int(send)])
            available_by_source[src.id] -= send
            target_allocated[dst.id] += send
            attacked_target_ids.add(dst.id)
            offense_sent_so_far += send

    return moves
